In [1]:
!pip install datasets openai

## 1. 데이터 전처리

In [61]:
import json
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from openai import OpenAI

In [62]:
# Load the dataset from Hugging Face
dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")

In [63]:
subset = dataset.select(range(100)).to_pandas()[['id', 'sql_prompt', 'sql_context', 'sql']]

In [64]:
subset.head()

,id,sql_prompt,sql_context,sql
0,5097,What is the total volume of timber sold by eac...,"CREATE TABLE salesperson (salesperson_id INT, ...","SELECT salesperson_id, name, SUM(volume) as to..."
1,5098,List all the unique equipment types and their ...,CREATE TABLE equipment_maintenance (equipment_...,"SELECT equipment_type, SUM(maintenance_frequen..."
2,5099,How many marine species are found in the South...,"CREATE TABLE marine_species (name VARCHAR(50),...",SELECT COUNT(*) FROM marine_species WHERE loca...
3,5100,What is the total trade value and average pric...,"CREATE TABLE trade_history (id INT, trader_id ...","SELECT trader_id, stock, SUM(price * quantity)..."
4,5101,Find the energy efficiency upgrades with the h...,"CREATE TABLE upgrades (id INT, cost FLOAT, typ...","SELECT type, cost FROM (SELECT type, cost, ROW..."


In [75]:
client = OpenAI(api_key="")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [66]:
system = '''## 지시사항
이 데이터는 text-to-sql 데이터입니다.
테이블 DDL과 실제 sql문을 참고하여 영어 prompt를 한글 prompt로 번역하세요.

시작!'''

In [67]:
user_prompt = []

for context, prompt, sql in \
  zip(subset['sql_context'].to_list(), subset['sql_prompt'].to_list(), subset['sql'].to_list()):
  user_prompt.append('테이블 DDL: ' + context + '\nSQL 쿼리: ' +  sql + '\n영어 텍스트: ' + prompt + '\n한글 텍스트:')

print(user_prompt[0])

테이블 DDL: CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
SQL 쿼리: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;
영어 텍스트: What is the total volume of timber sold by each salesperson, sorted by salesperson?
한글 텍스트:


In [68]:
result_lst = []
chkpoint = 1

for user in tqdm(user_prompt[:]):
  response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
      {"role": "system", "content": system},
      {"role": "user", "content": user}
    ],
    # temperature=0
  )
  result_lst.append(response.choices[0].message.content)

  chkpoint += 1
  if chkpoint %10 == 0:
    padded = result_lst + [None] * (len(subset) - len(result_lst))
    subset['korean_text'] = padded
    subset.to_csv('text-to-sql.csv', index=False, encoding='utf-8-sig')
    print(f'중간 저장 : {chkpoint}')


padded = result_lst + [None] * (len(subset) - len(result_lst))
subset['korean_text'] = padded
subset.to_csv('text-to-sql.csv', index=False, encoding='utf-8-sig')
print(f'최종 저장 완료')

  9%|▉         | 9/100 [00:08<01:31,  1.00s/it]

중간 저장 : 10


 19%|█▉        | 19/100 [00:16<01:01,  1.32it/s]

중간 저장 : 20


 29%|██▉       | 29/100 [00:25<01:05,  1.08it/s]

...

중간 저장 : 90


 99%|█████████▉| 99/100 [01:24<00:00,  1.45it/s]

중간 저장 : 100


100%|██████████| 100/100 [01:25<00:00,  1.17it/s]

최종 저장 완료


```python
result_lst = []

for user in tqdm(user_prompt):
  response = client.chat.completions.create(
    model="gpt-4-1106-preview",
    messages=[
      {"role": "system", "content": system},
      {"role": "user", "content": user}
    ],
    temperature=0
  )
  result_lst.append(response.choices[0].message.content)

subset['korean_text'] = result_lst
subset.to_csv('text-to-sql.csv', index=False, encoding='utf-8-sig')
```

## 2. LLM 학습 포맷으로 전처리

In [69]:
df = pd.read_csv('/content/text-to-sql.csv')

LLM은 입력과 출력의 형태로 학습 데이터가 구성된다. LLM에 입력으로 넣을 프롬프트와 LLM이 생성하기를 원하는 출력을 구성해보자.

In [70]:
instruction_lst = []
output_lst = []

for context, korean_text, sql in \
  zip(df['sql_context'].to_list(), df['korean_text'].to_list(), df['sql'].to_list()):
  instruction_lst.append('DDL statements:\n' + context + '\n입력 텍스트: ' + korean_text + '\n\n위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.')
  output_lst.append('쿼리 작성: ' + sql)

In [71]:
print(instruction_lst[0])

DDL statements:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
입력 텍스트: 각 판매원이 판매한 목재의 총량은 얼마이며, 판매원에 따라 정렬되어 있나요?

위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.


In [72]:
print(output_lst[0])

쿼리 작성: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;


In [73]:
df['instruction'] = instruction_lst
df['input'] = '' # 라마팩토리 형식을 맞춰주기 위해서 임의로 추가한 열
df['output'] = output_lst

In [74]:
# DataFrame에서 'instruction'과 'output' 열을 딕셔너리 리스트로 변환
data_to_save = df[['instruction', 'input', 'output']].to_dict(orient='records')

# JSON 파일로 저장
with open('text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=4)